# Phase 3: Legal Domain Fine-Tuning (QLoRA)

This notebook covers **two parallel legal adapters**:

| Variant | Dataset(s) | Focus | LoRA rank |
|---------|-----------|-------|----------|
| **3a — Legal Contracts** | `nguyen-brat/legal_contracts` + `pile-of-law/freelaw` | Contract clause analysis (universal) | r=16 |
| **3b — Indian Law** | `viber1/indian-law-dataset` + `InLegalNLI` + local BNS mapping | Indian law under BNS/BNSS/BSA (post-July 2024) | r=32 |

Set `PHASE` below to `"3a"` or `"3b"` before running. All later cells adapt automatically.

**GPU required:** T4 (16GB) — available free on Kaggle  
**Estimated runtime:** 3a ~60 min (3 epochs) | 3b ~75 min (4 epochs)

## Why two adapters?
- **3a** teaches universal contract clause patterns: indemnification, limitation of liability, force majeure, non-compete. Useful for any contract review task.
- **3b** teaches Indian-specific legal reasoning AND corrects post-July-2024 statute references (BNS/BNSS/BSA replaced IPC/CrPC/IEA). Public datasets are pre-2024 and teach repealed section numbers — the local BNS mapping dataset fixes this. Phase 3b is the foundation for JusticeAI (Project 9).

## 0. Configuration — choose 3a or 3b here

In [ ]:
# ─── CHANGE THIS ───────────────────────────────────────────────────────────
PHASE = "3a"   # "3a" = Legal Contracts  |  "3b" = Indian Law
# ───────────────────────────────────────────────────────────────────────────

assert PHASE in ("3a", "3b"), "PHASE must be '3a' or '3b'"
print(f"Running Phase {PHASE}: {'Legal Contracts' if PHASE == '3a' else 'Indian Law'}")

## 0. Kaggle Setup

Before running:
1. Enable GPU: **Settings → Accelerator → GPU T4 x1** (single GPU — avoid 2x T4 for QLoRA)
2. Enable Internet: **Settings → Internet → On**
3. Add HuggingFace token: **Add-ons → Secrets → HF_TOKEN**
4. If running Phase 3b, also add **GROQ_API_KEY** (for LLM-judge eval)

> **Phase 3b prerequisite:** You must have run `scripts/build_bns_mapping_dataset.py` locally
> and uploaded `data/bns_bnss_bsa_mapping.jsonl` to Kaggle as a dataset input.

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

## 1. Install Dependencies

In [ ]:
import subprocess, sys

pkgs = [
    "bitsandbytes==0.45.3",
    "transformers==4.46.3",
    "peft==0.13.2",
    "trl==0.12.2",
    "accelerate==1.1.1",
    "datasets==3.0.0",
    "huggingface_hub",
    "pyyaml",
    "groq",   # for LLM-as-judge eval
]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
    capture_output=True, text=True
)
print(result.stdout[-2000:] or "")
print(result.stderr[-500:] or "")
print("\nDone. Restart kernel if needed: Kernel → Restart → Run All.")

In [ ]:
import transformers, peft, trl, bitsandbytes, accelerate, datasets
print(f"transformers: {transformers.__version__}")
print(f"peft:         {peft.__version__}")
print(f"trl:          {trl.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print(f"datasets:     {datasets.__version__}")

## 2. Hyperparameters

In [ ]:
import torch

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

if PHASE == "3a":
    # ── Phase 3a: Legal Contracts ──────────────────────────────────────────
    DATASET_NAME        = "nguyen-brat/legal_contracts"
    AUXILIARY_DATASET   = "pile-of-law/pile-of-law"   # freelaw subset
    AUXILIARY_SAMPLES   = 5000
    AUXILIARY_WEIGHT    = 0.20                          # 20% of each batch
    BNS_MAPPING_PATH    = None
    INSTRUCTION_FIELD   = "instruction"
    CONTEXT_FIELD       = "context"                    # raw clause text
    RESPONSE_FIELD      = "output"
    LORA_R              = 16
    LORA_ALPHA          = 32
    LEARNING_RATE       = 1.5e-4
    NUM_EPOCHS          = 3
    WARMUP_RATIO        = 0.03
    SAVE_STEPS          = 150
    SAVE_TOTAL_LIMIT    = 2
    OUTPUT_DIR          = "/kaggle/working/phase3a-legal-contracts"
    HF_REPO             = "mistral-7b-legal-contracts-qlora"
    JUDGE_SYSTEM        = "legal_contracts"
    DOMAIN_BASELINE     = "~3.0/5.0"
    DOMAIN_TARGET       = "≥3.5/5.0"
else:
    # ── Phase 3b: Indian Law ───────────────────────────────────────────────
    DATASET_NAME        = "viber1/indian-law-dataset"
    AUXILIARY_DATASET   = "Exploration-Lab/InLegalNLI"
    AUXILIARY_SAMPLES   = 3000
    AUXILIARY_WEIGHT    = 0.25                          # 25% — high-signal Indian SC reasoning
    # Check a few common locations where the JSONL might be uploaded
    import os
    _candidates = [
        "/kaggle/input/bns-mapping/bns_bnss_bsa_mapping.jsonl",
        "/kaggle/working/bns_bnss_bsa_mapping.jsonl",
        "../data/bns_bnss_bsa_mapping.jsonl",
    ]
    BNS_MAPPING_PATH    = next((p for p in _candidates if os.path.exists(p)), None)
    INSTRUCTION_FIELD   = "question"
    CONTEXT_FIELD       = None
    RESPONSE_FIELD      = "answer"
    LORA_R              = 32                            # higher rank — larger domain gap
    LORA_ALPHA          = 64
    LEARNING_RATE       = 1.0e-4
    NUM_EPOCHS          = 4
    WARMUP_RATIO        = 0.05
    SAVE_STEPS          = 100
    SAVE_TOTAL_LIMIT    = 3
    OUTPUT_DIR          = "/kaggle/working/phase3b-indian-law"
    HF_REPO             = "mistral-7b-indian-law-qlora"
    JUDGE_SYSTEM        = "indian_law"
    DOMAIN_BASELINE     = "~2.5/5.0"
    DOMAIN_TARGET       = "≥3.5/5.0"

# ── Shared hyperparameters ─────────────────────────────────────────────────
LORA_DROPOUT   = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]
BATCH_SIZE          = 2
GRAD_ACCUM_STEPS    = 8    # effective batch = 16
MAX_SEQ_LENGTH      = 2048 # legal text is long
LOGGING_STEPS       = 10
TEST_SIZE           = 0.10
MAX_EVAL_SAMPLES    = 100  # Groq judge on 100 samples

print(f"Phase: {PHASE}")
print(f"Dataset:           {DATASET_NAME}")
print(f"LoRA rank:         {LORA_R}")
print(f"Epochs:            {NUM_EPOCHS}")
print(f"LR:                {LEARNING_RATE}")
print(f"BNS mapping:       {BNS_MAPPING_PATH or 'N/A'}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" \
      if torch.cuda.is_available() else "")

## 3. Load and Explore the Dataset

In [ ]:
from datasets import load_dataset

print(f"Loading primary dataset: {DATASET_NAME}...")
raw = load_dataset(DATASET_NAME, split="train")

print(f"Size:    {len(raw)} samples")
print(f"Columns: {raw.column_names}")
print()

for i in range(2):
    sample = raw[i]
    instr  = sample.get(INSTRUCTION_FIELD, "")
    ctx    = sample.get(CONTEXT_FIELD, "") if CONTEXT_FIELD else ""
    resp   = sample.get(RESPONSE_FIELD, "")
    print(f"─── Sample {i} ───")
    print(f"INSTRUCTION: {instr[:300]}")
    if ctx:
        print(f"CONTEXT:     {ctx[:300]}")
    print(f"RESPONSE:    {resp[:300]}")
    print()

In [ ]:
# Token length distribution — important for legal text (often long)
import numpy as np

def approx_tokens(sample):
    text = (sample.get(INSTRUCTION_FIELD) or "") + " " + \
           (sample.get(CONTEXT_FIELD) or "" if CONTEXT_FIELD else "") + " " + \
           (sample.get(RESPONSE_FIELD) or "")
    return len(text) // 4  # rough approximation: ~4 chars/token

lengths = [approx_tokens(s) for s in raw]
print(f"Approx token length stats (instruction + context + response):")
print(f"  median: {int(np.median(lengths))}")
print(f"  p90:    {int(np.percentile(lengths, 90))}")
print(f"  p99:    {int(np.percentile(lengths, 99))}")
print(f"  max:    {max(lengths)}")
print(f"  MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} covers p90: {'yes' if int(np.percentile(lengths, 90)) < MAX_SEQ_LENGTH else 'no'}")

## 4. Load Tokenizer and Format Dataset

In [ ]:
from transformers import AutoTokenizer

print(f"Loading tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "right"
print(f"Vocab size: {tokenizer.vocab_size}")

In [ ]:
LEGAL_CONTRACTS_SYSTEM = (
    "You are an expert legal assistant specializing in contract law. "
    "When given a clause from a legal contract, analyze it carefully and provide "
    "a clear, accurate explanation of its legal implications, risks, and key terms. "
    "Use precise legal language while remaining accessible to non-lawyers. "
    "Cite relevant legal concepts and flag any unusual or one-sided provisions."
)

INDIAN_LAW_SYSTEM = (
    "You are an expert in Indian law with deep knowledge of the Bharatiya Nyaya Sanhita "
    "(BNS, 2023), Bharatiya Nagarik Suraksha Sanhita (BNSS, 2023), Bharatiya Sakshya "
    "Adhiniyam (BSA, 2023), the Constitution of India, and Indian Supreme Court and High "
    "Court jurisprudence. When answering legal questions, cite the correct current statutes "
    "(BNS/BNSS/BSA for post-July 2024 matters, IPC/CrPC/IEA for historical cases), "
    "reference relevant Supreme Court precedents where applicable, and provide accurate, "
    "precise legal analysis. Clearly distinguish between the old codes (IPC/CrPC) and the "
    "new codes (BNS/BNSS/BSA) when the distinction is legally material."
)

SYSTEM_PROMPT = LEGAL_CONTRACTS_SYSTEM if PHASE == "3a" else INDIAN_LAW_SYSTEM
print(f"System prompt set for Phase {PHASE}")
print(f"Preview: {SYSTEM_PROMPT[:200]}...")

In [ ]:
def format_legal_sample(sample: dict) -> str:
    instruction = sample.get(INSTRUCTION_FIELD, "") or ""
    context     = sample.get(CONTEXT_FIELD, "") or "" if CONTEXT_FIELD else ""
    response    = sample.get(RESPONSE_FIELD, "") or ""

    # Merge instruction + context if both are present (Phase 3a)
    if context and context.strip():
        user_msg = f"{instruction}\n\nClause:\n{context}"
    else:
        user_msg = instruction

    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": response},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

# Test on one sample
sample_text = format_legal_sample(raw[0])
print(f"Example (first 600 chars):\n{sample_text[:600]}")
print(f"\nToken count: {len(tokenizer.encode(sample_text))}")

In [ ]:
split     = raw.train_test_split(test_size=TEST_SIZE, seed=42)
train_raw = split["train"]
test_raw  = split["test"]

print(f"Train: {len(train_raw)} | Test: {len(test_raw)}")

train_primary = train_raw.map(
    lambda s: {"text": format_legal_sample(s)},
    remove_columns=train_raw.column_names,
)
print(f"Formatted primary training set: {len(train_primary)} samples")

In [ ]:
# ── Load and format auxiliary dataset ─────────────────────────────────────
from datasets import concatenate_datasets

print(f"Loading auxiliary dataset: {AUXILIARY_DATASET} ({AUXILIARY_SAMPLES} samples)...")

if PHASE == "3a":
    # pile-of-law/freelaw — legal prose pretraining signal
    aux_raw = load_dataset(AUXILIARY_DATASET, "freelaw", split="train",
                           streaming=False).select(range(AUXILIARY_SAMPLES))

    def format_freelaw(sample: dict) -> str:
        text = sample.get("text") or sample.get("content") or ""
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": "Read and summarize the following court opinion excerpt."},
            {"role": "assistant", "content": text[:1800]},
        ]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )

    train_aux = aux_raw.map(
        lambda s: {"text": format_freelaw(s)},
        remove_columns=aux_raw.column_names,
    )

else:
    # InLegalNLI — Indian SC judgment NLI pairs
    aux_raw = load_dataset(AUXILIARY_DATASET, split="train").select(range(AUXILIARY_SAMPLES))

    def format_inlegalnli(sample: dict) -> str:
        premise   = sample.get("premise") or ""
        hypothesis = sample.get("hypothesis") or ""
        label     = sample.get("label", "")
        label_str = {0: "entailment", 1: "neutral", 2: "contradiction"}.get(label, str(label))
        user_msg  = (
            f"Judgment excerpt:\n{premise}\n\n"
            f"Claim: {hypothesis}\n\n"
            f"Does the judgment support, contradict, or remain neutral to the claim? "
            f"Explain with reference to Indian law."
        )
        resp = (
            f"The relationship is: {label_str}. "
            f"The judgment {'supports' if label_str == 'entailment' else 'does not directly support'} "
            f"the claim based on the stated legal reasoning."
        )
        messages = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": resp},
        ]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )

    train_aux = aux_raw.map(
        lambda s: {"text": format_inlegalnli(s)},
        remove_columns=aux_raw.column_names,
    )

print(f"Auxiliary samples formatted: {len(train_aux)}")

In [ ]:
# ── Load and format BNS mapping (Phase 3b only) ────────────────────────────
import json
import os

train_bns = None

if PHASE == "3b":
    if BNS_MAPPING_PATH and os.path.exists(BNS_MAPPING_PATH):
        bns_samples = []
        with open(BNS_MAPPING_PATH) as f:
            for line in f:
                line = line.strip()
                if line:
                    bns_samples.append(json.loads(line))
        print(f"BNS mapping samples loaded: {len(bns_samples)}")

        from datasets import Dataset as HFDataset

        def format_bns(sample: dict) -> str:
            messages = [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": sample.get("instruction", "")},
                {"role": "assistant", "content": sample.get("output", "")},
            ]
            return tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )

        bns_ds   = HFDataset.from_list(bns_samples)
        train_bns = bns_ds.map(
            lambda s: {"text": format_bns(s)},
            remove_columns=bns_ds.column_names,
        )
        print(f"BNS mapping formatted: {len(train_bns)} samples")
    else:
        print("WARNING: BNS mapping JSONL not found.")
        print("Run: python scripts/build_bns_mapping_dataset.py")
        print("Then upload data/bns_bnss_bsa_mapping.jsonl as a Kaggle dataset input.")
        print("Training will proceed WITHOUT BNS correction — model may cite IPC sections.")

In [ ]:
# ── Merge datasets with weighted mixing ────────────────────────────────────
# Primary: 1 - auxiliary_weight - bns_weight
# Auxiliary: auxiliary_weight
# BNS (3b only): 0.15

import math
from datasets import concatenate_datasets

bns_weight  = 0.15 if (PHASE == "3b" and train_bns) else 0.0
primary_weight = 1.0 - AUXILIARY_WEIGHT - bns_weight

n_primary   = len(train_primary)
n_aux_target = math.ceil(n_primary * (AUXILIARY_WEIGHT / primary_weight))
n_aux_target = min(n_aux_target, len(train_aux))

aux_subset = train_aux.select(range(n_aux_target))

parts = [train_primary, aux_subset]
part_labels = [f"primary ({len(train_primary)})", f"auxiliary ({len(aux_subset)})"]

if train_bns is not None:
    n_bns_target = math.ceil(n_primary * (bns_weight / primary_weight))
    # BNS mapping is small — repeat it to hit the target weight
    from datasets import Dataset as HFDataset
    if n_bns_target > len(train_bns):
        repeats   = math.ceil(n_bns_target / len(train_bns))
        train_bns_rep = concatenate_datasets([train_bns] * repeats)
        train_bns_rep = train_bns_rep.select(range(n_bns_target))
    else:
        train_bns_rep = train_bns.select(range(n_bns_target))
    parts.append(train_bns_rep)
    part_labels.append(f"BNS mapping ({len(train_bns_rep)})")

train_dataset = concatenate_datasets(parts).shuffle(seed=42)
print(f"Final training set: {len(train_dataset)} samples")
for label in part_labels:
    print(f"  {label}")

## 5. Load Model in 4-bit + Inject LoRA

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {BASE_MODEL} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print(f"Model loaded. GPU: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Phase 3a: r=16 → ~20M trainable params (0.53%)
# Phase 3b: r=32 → ~40M trainable params (1.07%)

## 6. Train

In [ ]:
import time
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    fp16=False,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    report_to="none",
    optim="paged_adamw_32bit",
    gradient_checkpointing=True,
    group_by_length=True,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)

eff_batch = BATCH_SIZE * GRAD_ACCUM_STEPS
steps_per_epoch = len(train_dataset) // eff_batch
print(f"Training {len(train_dataset)} samples × {NUM_EPOCHS} epochs")
print(f"Effective batch: {eff_batch} | Steps/epoch: ~{steps_per_epoch}")
print()

t0 = time.time()
trainer.train()
print(f"\nTraining time: {(time.time() - t0) / 60:.1f} min")

In [ ]:
adapter_path = f"{OUTPUT_DIR}/final-adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved: {adapter_path}")

import os
total_size = sum(
    os.path.getsize(os.path.join(adapter_path, f))
    for f in os.listdir(adapter_path)
    if os.path.isfile(os.path.join(adapter_path, f))
)
print(f"Adapter size: {total_size / 1e6:.1f} MB")

## 7. Training Loss Curve

In [ ]:
import matplotlib.pyplot as plt

log_history   = trainer.state.log_history
train_losses  = [(e['step'], e['loss']) for e in log_history if 'loss' in e]

if train_losses:
    steps, losses = zip(*train_losses)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(steps, losses, color='steelblue', linewidth=1.5, alpha=0.8)
    ax.set_xlabel('Step')
    ax.set_ylabel('Training Loss')
    ax.set_title(f'Phase {PHASE} QLoRA — Training Loss')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/loss_curve.png", dpi=150)
    plt.show()
    print(f"Initial: {losses[0]:.4f} | Final: {losses[-1]:.4f} | Reduction: {(losses[0]-losses[-1])/losses[0]:.1%}")

## 8. Qualitative Inference Check

In [ ]:
model.eval()

def generate_legal(question: str, max_new_tokens: int = 400) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs    = tokenizer(formatted, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()

# Phase-appropriate test question
if PHASE == "3a":
    test_q = (
        "Analyze this indemnification clause: "
        "'The Vendor shall indemnify and hold harmless the Client from any and all claims "
        "arising from the Vendor's performance of services under this Agreement, "
        "except to the extent caused by the Client's own negligence.'"
    )
else:
    test_q = (
        "Under the Bharatiya Nyaya Sanhita 2023, what is the punishment for culpable homicide "
        "not amounting to murder? Which section applies, and how does it differ from the "
        "equivalent provision under the repealed Indian Penal Code?"
    )

print(f"Question:\n{test_q}\n")
print(f"Response:\n{generate_legal(test_q)}")

## 9. LLM-as-Judge Evaluation (Groq)

Scores 100 test-split responses on a 1–5 scale using `llama-3.3-70b-versatile` via Groq.

- **Phase 3a rubric**: contract clause correctness, legal precision, risk identification
- **Phase 3b rubric**: correct BNS/BNSS/BSA vs IPC/CrPC citations, procedural accuracy

Groq free tier: ~14,400 requests/day — 100 samples runs in ~2 minutes.

In [ ]:
import os, re

try:
    from kaggle_secrets import UserSecretsClient
    groq_key = UserSecretsClient().get_secret("GROQ_API_KEY")
    print("GROQ_API_KEY loaded from Kaggle secrets.")
except Exception:
    groq_key = os.environ.get("GROQ_API_KEY") or input("Enter GROQ_API_KEY: ").strip()

from groq import Groq
groq_client = Groq(api_key=groq_key)
JUDGE_MODEL = "llama-3.3-70b-versatile"

RUBRICS = {
    "legal_contracts": (
        "Rating criteria for contract clause responses:\n"
        "1 — Wrong, irrelevant, or legally incorrect\n"
        "2 — Partially correct with significant legal errors or omissions\n"
        "3 — Correct interpretation but missing key legal implications\n"
        "4 — Correct, complete, minor omissions; good legal precision\n"
        "5 — Accurate, complete, legally precise; correctly identifies risks and obligations"
    ),
    "indian_law": (
        "Rating criteria for Indian law responses:\n"
        "1 — Wrong statute cited, incorrect legal principle, or factually wrong\n"
        "2 — Correct general area but wrong section numbers or significant procedural errors\n"
        "3 — Correct answer but missing key nuance (e.g. old vs new code distinction)\n"
        "4 — Correct, proper citations, minor omissions in nuance or case law\n"
        "5 — Accurate, correct BNS/BNSS/BSA vs IPC/CrPC distinction, relevant SC precedents cited"
    ),
}

rubric = RUBRICS[JUDGE_SYSTEM]
print(f"Rubric: {JUDGE_SYSTEM}")

In [ ]:
from tqdm.auto import tqdm

eval_samples = test_raw.select(range(min(MAX_EVAL_SAMPLES, len(test_raw))))
scores, records = [], []

for sample in tqdm(eval_samples, desc="LLM-judge eval"):
    instruction = sample.get(INSTRUCTION_FIELD, "") or ""
    context     = sample.get(CONTEXT_FIELD, "") or "" if CONTEXT_FIELD else ""
    gt_response = sample.get(RESPONSE_FIELD, "") or ""

    user_prompt = f"{instruction}\n\nClause:\n{context}" if context else instruction
    response    = generate_legal(user_prompt)

    judge_prompt = (
        f"You are an expert evaluator. Rate the following AI response on a scale of 1-5.\n\n"
        f"Question: {instruction[:400]}\n\n"
        f"Reference answer: {gt_response[:400]}\n\n"
        f"AI response: {response[:400]}\n\n"
        f"{rubric}\n\n"
        f"Respond with ONLY a single digit (1-5). No explanation."
    )

    try:
        completion = groq_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": judge_prompt}],
            max_tokens=5,
            temperature=0,
        )
        score_text = completion.choices[0].message.content or ""
        m = re.search(r'[1-5]', score_text.strip())
        score = int(m.group()) if m else 3
    except Exception as e:
        print(f"Judge error: {e}")
        score = 3

    scores.append(score)
    records.append({"instruction": instruction[:100], "score": score, "response": response[:200]})

avg_score  = sum(scores) / len(scores) if scores else 0.0
score_dist = {str(i): scores.count(i) for i in range(1, 6)}

print(f"\n{'='*50}")
print(f"LLM JUDGE RESULTS (Groq {JUDGE_MODEL})")
print(f"Average score: {avg_score:.2f}/5.0")
print(f"Distribution:  {score_dist}")
print(f"Baseline:      {DOMAIN_BASELINE}")
print(f"Target:        {DOMAIN_TARGET}")
print(f"{'='*50}")

In [ ]:
import json

eval_summary = {
    "phase":        f"phase{PHASE}_legal",
    "model":        BASE_MODEL,
    "adapter":      adapter_path,
    "dataset":      DATASET_NAME,
    "lora_r":       LORA_R,
    "num_epochs":   NUM_EPOCHS,
    "avg_score":    round(avg_score, 3),
    "score_dist":   score_dist,
    "baseline":     DOMAIN_BASELINE,
    "target":       DOMAIN_TARGET,
}

with open(f"{OUTPUT_DIR}/eval_results.json", "w") as f:
    json.dump(eval_summary, f, indent=2)

print(json.dumps(eval_summary, indent=2))

## 10. Push Adapter to HuggingFace Hub

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets.")
except Exception:
    hf_token = os.environ.get("HF_TOKEN") or input("Enter HF_TOKEN: ").strip()

from huggingface_hub import login, HfApi
login(token=hf_token)

HF_USERNAME = "anksriv"  # update if different
REPO_ID     = f"{HF_USERNAME}/{HF_REPO}"

api = HfApi()
api.create_repo(repo_id=REPO_ID, exist_ok=True, private=False)

trainer.model.push_to_hub(REPO_ID, token=hf_token)
tokenizer.push_to_hub(REPO_ID, token=hf_token)

api.upload_file(
    path_or_fileobj=f"{OUTPUT_DIR}/eval_results.json",
    path_in_repo="eval_results.json",
    repo_id=REPO_ID,
    token=hf_token,
)

print(f"\nAdapter pushed: https://huggingface.co/{REPO_ID}")

## 11. Summary

| Item | Phase 3a | Phase 3b |
|------|----------|----------|
| Base model | Mistral-7B-Instruct-v0.2 | Mistral-7B-Instruct-v0.2 |
| Primary dataset | nguyen-brat/legal_contracts | viber1/indian-law-dataset |
| Auxiliary | pile-of-law/freelaw (5K) | InLegalNLI (3K) + BNS mapping (~800) |
| LoRA rank | r=16 | r=32 |
| Epochs | 3 | 4 |
| Max seq length | 2048 | 2048 |
| Eval metric | LLM-judge 1–5 | LLM-judge 1–5 (Indian law rubric) |
| Baseline | ~3.0/5.0 | ~2.5/5.0 |
| Target | ≥3.5/5.0 | ≥3.5/5.0 |

**Phase 3b note**: The BNS mapping component (`data/bns_bnss_bsa_mapping.jsonl`) is what prevents the model from citing IPC §302 (repealed July 2024) instead of BNS §103. Without it, the adapter would confidently produce phantom citations — the exact failure mode flagged by the Indian Supreme Court in early-2026 AI legal tool reviews.

**Next step:** Run `notebooks/04_finance.ipynb`.